# 13 — External Candidate Coverage Gate

**Purpose.** Measure whether held-out structures are reachable from EXTERNAL structure sources (COCONUT now, PubChem as a membership estimate) under a leakage-free protocol.

* **Coverage only.** Nothing is trained, tuned or ranked here. No Recall@25 / Recall@100 / MRR from any ranker is a Gate-A number. Ranking of external candidates starts in notebook 14 (analog baseline).
* **Read-only.** It reads the finalized universe `candidates/universe/` (TRAIN ∪ COCONUT), the active Stage-A builds `candidates/universe_work/`, the grouped validation queries and the optional PubChem membership index. It never rebuilds the universe; notebook 12 is NOT re-run.
* **Outputs.** Everything goes to `reports/candidate_coverage/` on Drive (`coverage.report_dir`).

Run order: notebook 10 (fresh runtime) → **this notebook** → follow the decision printed at the end.

## Why the old C2 Recall@All could be trivial (leakage explanation)

Validation molecules come from TRAIN. The finalized universe is TRAIN ∪ COCONUT, so:

1. **TRAIN provenance.** Every held-out truth is in the universe through its TRAIN record. "Truth in universe" (old `universe` mode) is ~100% by construction. It measures mass accuracy, not database coverage.
2. **Selection on the outcome.** The old C2 regime was *selected* on having a non-TRAIN source (`class2_split.class2_eligible`). C2 `external_only` Recall@All was therefore conditioned on the property it was meant to measure.

### External-only protocol (this notebook)

For **every** held-out truth connectivity `k` of the grouped validation split (all of them, not a reachability-selected subset):

`candidate_sources_without_train(k) = candidate_sources(k) − {TRAIN}`

| label | meaning |
|---|---|
| `TRAIN_REACHABLE` | k has TRAIN provenance: **diagnostic only, never counted** |
| `COCONUT_REACHABLE` | k has COCONUT provenance (finalized universe, current filter policy) |
| `PUBCHEM_REACHABLE` | k is in the local PubChem key-membership index: **PUBCHEM_KEY_MEMBERSHIP_ESTIMATE** (PubChem's stored InChIKey block, not guaranteed canonical parity) |
| `EXTERNAL_REACHABLE` | any external source remains after removing TRAIN |
| `EXTERNAL_UNREACHABLE` | none remains: TRAIN-only truth |

**C2 (external protocol)** = held-out truths that ARE externally reachable. A TRAIN-only truth is never C2. No 696k universe is rebuilt per query: TRAIN provenance is masked per truth, and the external-only pool is one boolean mask over candidate ids.

**Database coverage failure ≠ mass retrieval failure.** Not externally reachable means fix the DATABASE. Reachable but outside the ppm window means fix precursor, adduct or mass handling.

The old ranking diagnostics of `13_colab_candidate_recall.ipynb` remain **LEGACY_DIAGNOSTIC_ONLY** and are never a Gate-A decision.

## Decision rules: declared BEFORE any result is computed

Thresholds come from `coverage.decision` in `configs/casmi_v2_colab.yaml`. The next code cell writes them to `decision_rules.json` *before* any coverage number exists. Rules are evaluated mechanically at the end. **Do not tune thresholds after seeing results.**

| rule | condition (natural-product panel = `coverage.decision.natural_product_panel`) | decision |
|---|---|---|
| **A** | PubChem adds ≥ `rule_a_min_incremental_pp` (10) percentage points of external reachability over COCONUT | `BUILD_PUBCHEM_TIER1_NEXT` |
| **B** | reachability ≥ `rule_b_min_reachability` AND the `rule_b_statistic` of `rule_b_pool` at ±`rule_b_ppm` ppm ≥ `rule_b_large_pool_threshold` | `BUILD_FORMULA_PRUNING_BEFORE_RANKING` |
| **C** | charge / salt rescue recovers ≥ `rule_c_min_rescued_fraction` of the truths reachable before filters (only with Stage-B/C invalidation fixed, which it is on this branch) | `REBUILD_COCONUT_WITH_RESCUE_POLICY` |
| **D** | COCONUT reachability ≥ `rule_d_min_coconut_reachability` AND filter-lost fraction ≤ `rule_d_max_lost_fraction` | `PROCEED_TO_ANALOG_BASELINE` |

* Only Rule A's 10 pp threshold is given by the project. "Very large", "meaningful", "high" and "immaterial" have **no** project threshold. They default to `null`, and that rule is reported **NOT_EVALUATED** until you set it before running.
* Several rules may fire. The decision is the first fired rule in `priority` (default A, C, B, D: coverage fixes before pool density before ranking).
* No external source in the universe, or no validation population, gives `PROTOCOL_INVALID`. If nothing fires but a rule was not evaluated, the decision is `UNDETERMINED_THRESHOLDS_OR_INPUTS_MISSING`.

In [ ]:
# bootstrap: Drive, repository (cloned by notebook 10), config, seeds -- nothing is computed yet
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
# repository: ENVEDA_REPO_ROOT, else the notebook-10 clone, else a Drive copy
_repo_candidates = [os.environ.get('ENVEDA_REPO_ROOT'), str(Path('/content', 'Enveda')), str(DRIVE_ROOT / 'repo')]
REPO_ROOT = next((Path(p) for p in _repo_candidates if p and (Path(p) / 'src' / 'casmi').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'CASMI repository not found in {_repo_candidates}: run notebook 10 (it clones the repository) or set ENVEDA_REPO_ROOT')
os.environ['ENVEDA_REPO_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report
ensure_packages({'rdkit': 'rdkit', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '13_candidate_coverage'
COV = CFG['coverage']
REPORT = P.drive_root / COV['report_dir']
REPORT.mkdir(parents=True, exist_ok=True)
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
WORK = P.candidate_db_dir / 'universe_work'
print('repo', REPO_ROOT, '| scratch', P.scratch_dir, '| reports', REPORT)
print(system_report())

In [ ]:
# decision policy fixed and SAVED before any coverage number exists (pre-registration)
from datetime import datetime, timezone
from casmi.validation.external_coverage import COVERAGE_GATE_VERSION, DecisionPolicy, write_json
POLICY = DecisionPolicy.from_dict(COV['decision'])
DECISION_RULES = {
    'coverage_gate_version': COVERAGE_GATE_VERSION, 'declared_at': datetime.now(timezone.utc).isoformat(),
    'policy': dict(POLICY.__dict__), 'policy_identity': POLICY.identity(),
    'rules': {'A': 'PubChem adds >= rule_a_min_incremental_pp pp of external reachability over COCONUT on the NP panel -> BUILD_PUBCHEM_TIER1_NEXT',
              'B': 'reachability >= rule_b_min_reachability AND pool statistic at rule_b_ppm >= rule_b_large_pool_threshold -> BUILD_FORMULA_PRUNING_BEFORE_RANKING',
              'C': 'rescue recovers >= rule_c_min_rescued_fraction of truths reachable before filters -> REBUILD_COCONUT_WITH_RESCUE_POLICY',
              'D': 'COCONUT reachability >= rule_d_min_coconut_reachability AND lost fraction <= rule_d_max_lost_fraction -> PROCEED_TO_ANALOG_BASELINE'},
    'unset_threshold_behaviour': 'NOT_EVALUATED (never guessed)'}
write_json(REPORT / 'decision_rules.json', DECISION_RULES)
print(json.dumps(DECISION_RULES, indent=2, default=str))

In [ ]:
# finalized universe status (read-only; legacy manifests stay readable)
from casmi.candidates.universe import universe_manifest_status, universe_source_status
if not (UNIVERSE_ROOT / 'universe_manifest.json').exists():
    raise FileNotFoundError(f'no finalized universe under {UNIVERSE_ROOT}: run notebook 12 (Stage A / B / C) once')
USTATUS = universe_source_status(UNIVERSE_ROOT)
MSTATUS = universe_manifest_status(UNIVERSE_ROOT)
print(json.dumps({'universe_status': USTATUS['status'], 'n_candidates': USTATUS.get('n_candidates'), **MSTATUS}, indent=2, default=str))
if MSTATUS['legacy_manifest_detected']:
    print('legacy_manifest_detected = true: built before manifest v3. READ-ONLY coverage use is valid; an explicit rebuild '
          '(notebook 12 Stage B + C, Stage A reused) is required only before identity-dependent reuse.')
if USTATUS['status'] != 'external':
    print('*** no external source in the finalized universe: the external protocol is INVALID (decision PROTOCOL_INVALID)')

In [ ]:
# validation populations (grouped connectivity split; panels are never merged silently)
import numpy as np, pandas as pd
from casmi.validation.external_coverage import PanelSpec, build_population_table, molecule_table, read_optional_table
DEV_COLS = ['query_id', 'true_connectivity_key', 'fold', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type', 'exact_mass_true']
dev_path = input_path(CFG, P, 'dev_queries')
if not dev_path.exists():
    raise FileNotFoundError(f'grouped validation queries missing: {dev_path} (upload data/interim/candidate_generation/dev_queries.parquet)')
dev = pd.read_parquet(dev_path)
missing = {'query_id', 'true_connectivity_key', 'fold', 'neutral_mass'} - set(dev.columns)
if missing:
    raise KeyError(f'dev_queries lacks required columns {sorted(missing)}')
dev = dev[[c for c in DEV_COLS if c in dev.columns]]
REG_PATH = P.processed_dir / 'validation_regimes.parquet'
REG = pd.read_parquet(REG_PATH, columns=['query_id', 'regime']) if REG_PATH.exists() else None
print('C1/C2/C3 manifest:', REG_PATH if REG is not None else 'not found (regime column = None)')
PANELS = [PanelSpec.from_dict(n, d) for n, d in (COV.get('panels') or {}).items()]
FRAMES = {p.name: read_optional_table(P.drive_root / p.path) for p in PANELS if p.kind == 'file' and p.path}
Q = build_population_table(dev, REG, PANELS, FRAMES)
MOL = molecule_table(Q)
display(Q.groupby('panel').agg(n_queries=('query_id', 'size'), n_molecules=('connectivity_key', 'nunique')))
if 'instrument_type' in dev.columns:
    display(dev['instrument_type'].value_counts().head(15))
for p in PANELS:
    if not (Q['panel'] == p.name).any():
        print(f'WARNING: panel {p.name!r} is EMPTY: check coverage.panels.{p.name} (values {p.values}) against the counts above')

In [ ]:
# membership + leakage-free reachability (TRAIN provenance removed for every held-out truth)
from casmi.candidates.pubchem_membership import PubChemMembershipIndex, membership_summary
from casmi.validation.external_coverage import (build_external_membership, c2_external_protocol, classify_external_reachability,
                                                 stratify_by_mass_bin, universe_truth_rows)
TRUTH_ROWS = universe_truth_rows(UNIVERSE_ROOT, MOL['connectivity_key'])
PC_DIR = P.drive_root / COV['pubchem_membership']['index_dir']
PCI = PubChemMembershipIndex(PC_DIR) if PubChemMembershipIndex.available(PC_DIR) else None
print('PubChem membership:', 'AVAILABLE (PUBCHEM_KEY_MEMBERSHIP_ESTIMATE)' if PCI else 'NOT_AVAILABLE (PubChem columns stay empty)')
MEMB = classify_external_reachability(build_external_membership(MOL['connectivity_key'], TRUTH_ROWS, PCI))
MOL = c2_external_protocol(MOL.merge(MEMB, on='connectivity_key', how='left', validate='many_to_one'))
MOL['mass_bin'] = stratify_by_mass_bin(MOL['truth_exact_mass'].fillna(MOL['median_query_neutral_mass']), CFG['validation']['mass_bins_da'])
MOL['truth_formula'] = MOL['connectivity_key'].map(TRUTH_ROWS.set_index('connectivity_key')['molecular_formula'])
print('legacy C2 truths without any external source (must be 0):', int(MOL['legacy_c2_train_only'].sum()))
diag = MOL.groupby('panel')[['in_train', 'membership_in_full_train_union', 'external_reachable']].mean()
display(diag.rename(columns={'in_train': 'train_membership_DIAGNOSTIC', 'membership_in_full_train_union': 'in_full_universe_DIAGNOSTIC_NOT_COVERAGE'}))

In [ ]:
# external reachability report: panel / validation regime / mass bin (TRAIN never counts)
from casmi.validation.external_coverage import reachability_summary
REACH_PANEL = reachability_summary(MOL, by=('panel',))
REACH_REGIME = reachability_summary(MOL.assign(regime=MOL['regime'].fillna('UNASSIGNED')), by=('panel', 'regime'))
REACH_MASS = reachability_summary(MOL, by=('panel', 'mass_bin'))
display(REACH_PANEL)
display(REACH_REGIME)
MOL_OUT = MOL.assign(candidate_sources=MOL['candidate_sources'].map(';'.join),
                     candidate_sources_without_train=MOL['candidate_sources_without_train'].map(';'.join))
MOL_OUT.to_parquet(REPORT / 'external_reachability_by_molecule.parquet', index=False)
pd.concat([REACH_PANEL.assign(grouping='panel'), REACH_REGIME.assign(grouping='panel+regime')], ignore_index=True).to_csv(
    REPORT / 'external_reachability_summary.csv', index=False)
REACH_MASS.to_csv(REPORT / 'reachability_by_mass_bin.csv', index=False)

In [ ]:
# optional PubChem key membership (local index only; NOT_AVAILABLE without it)
PC_SUMMARY = membership_summary(PCI, MOL['connectivity_key'].unique() if PCI else None)
INCR = REACH_MASS[['panel', 'mass_bin', 'n_molecules', 'coconut_reachability', 'pubchem_reachability', 'union_external_reachability',
                   'incremental_pubchem_over_coconut_pp', 'pubchem_label']]
write_json(REPORT / 'pubchem_membership_summary.json', PC_SUMMARY)
INCR.to_csv(REPORT / 'pubchem_incremental_coverage.csv', index=False)
print(json.dumps(PC_SUMMARY, indent=2, default=str))
display(INCR)

In [ ]:
# mass-window coverage at 2 / 5 / 10 ppm: database coverage vs mass retrieval (no ranking)
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.pubchem_tier1 import Tier1MassIndex
from casmi.validation.external_coverage import evaluate_mass_window_coverage, external_candidate_mask, window_outcome_summary
IDX = CandidateMassIndex.load(UNIVERSE_ROOT / 'index')
MASKS = external_candidate_mask(UNIVERSE_ROOT)
print({k: int(v.sum()) for k, v in MASKS.items()}, '| candidates', IDX.n_candidates)
TIER1_TABLE = P.drive_root / COV['pubchem_tier1']['table']
TIER1 = Tier1MassIndex.from_table(TIER1_TABLE) if TIER1_TABLE.exists() else None
print('projected PubChem Tier-1 pools:', 'AVAILABLE (RAW CID counts, not connectivities)' if TIER1 else 'NOT_AVAILABLE')
cid_of = TRUTH_ROWS.set_index('connectivity_key')['candidate_id']
QW = Q.merge(MOL[['panel', 'connectivity_key', 'external_reachable', 'mass_bin']], on=['panel', 'connectivity_key'], how='left')
QW['truth_candidate_id'] = QW['connectivity_key'].map(cid_of).fillna(-1).astype(np.int64)
WIN = evaluate_mass_window_coverage(IDX, QW, COV['ppm_values'], external_mask=MASKS['external'], tier1_index=TIER1)
WIN = WIN.merge(QW[['panel', 'query_id', 'mass_bin', 'regime']], on=['panel', 'query_id'], how='left')
WIN_SUMMARY = window_outcome_summary(WIN)
display(WIN_SUMMARY)

In [ ]:
# candidate-pool density: finalized connectivity counts vs projected RAW PubChem Tier-1 counts (never added together)
from casmi.validation.external_coverage import summarize_pool_density
POOL = summarize_pool_density(WIN, by=('panel', 'ppm'))
display(POOL[POOL['n'] > 0])
WIN.to_parquet(REPORT / 'mass_pool_density.parquet', index=False)
POOL.to_csv(REPORT / 'mass_pool_density_summary.csv', index=False)
WIN_SUMMARY.to_csv(REPORT / 'mass_window_outcomes.csv', index=False)
FAIL_KIND = WIN.groupby(['panel', 'ppm'])['coverage_outcome'].value_counts(normalize=True).unstack(fill_value=0)
print('database_coverage_failure = NOT externally reachable  -> fix the DATABASE')
print('mass_retrieval_failure    = reachable, outside the ppm window -> fix precursor / adduct / MASS handling')
display(FAIL_KIND.round(4))

In [ ]:
# filter-loss audit inputs: external records the CURRENT policy dropped (read-only, active Stage-A builds)
from casmi.candidates.stage_a import StructureEngine, active_stage_a_builds
from casmi.validation.external_coverage import collect_lost_records, key_unkeyed_lost_records
from casmi.workspace.parallel import resolve_n_jobs
ACTIVE = active_stage_a_builds(WORK)
print({k: (v['status'], v['build_id']) for k, v in ACTIVE.items()})
NS = [v['namespace'] for v in ACTIVE.values() if v['status'] == 'complete']
LOST = collect_lost_records(NS)
display(LOST.groupby(['source', 'loss_stage', 'loss_categories']).size().rename('n').reset_index())
N_JOBS = resolve_n_jobs(COV['rescue']['n_jobs'])
if COV['rescue']['canonicalize_prefiltered'] and len(LOST):
    with StructureEngine('universe_minimal', N_JOBS) as ENG:      # training canonicalizer, current policy (no cleanup)
        LOST = key_unkeyed_lost_records(LOST, ENG)
n_keyed = int(LOST['connectivity_key'].notna().sum()) if len(LOST) else 0
print(f'lost records: {len(LOST):,} | with a connectivity key: {n_keyed:,} | workers {N_JOBS}')

In [ ]:
# current_policy vs rescue_policy (Uncharger + fragment parent + retained permanent charges); production universe untouched
from casmi.candidates.universe import UniverseBuildConfig
from casmi.validation.external_coverage import audit_filter_losses, filter_loss_summary, filter_policy_comparison, rescue_lost_records
UCFG = UniverseBuildConfig.from_dict(CFG['universe'])
with StructureEngine('universe_minimal', N_JOBS) as ENG:
    RESCUED = rescue_lost_records(LOST, UCFG.filters, ENG, policy='rescue_policy')
if len(RESCUED):
    display(RESCUED.groupby(['rescue_kind', 'passes_after_rescue']).size().rename('n').reset_index())
EXT_SOURCES = sorted(s for s, v in (USTATUS.get('sources') or {}).items() if s != 'TRAIN' and (v or {}).get('present'))
LOSS = audit_filter_losses(MOL, LOST, RESCUED, external_sources=EXT_SOURCES, mass_bins=CFG['validation']['mass_bins_da'])
LOSS_SUMMARY = filter_loss_summary(LOSS)
POLICY_CMP = filter_policy_comparison(LOSS)
display(LOSS_SUMMARY)
display(POLICY_CMP)
if len(LOSS):
    LOSS.to_parquet(REPORT / 'filter_loss_by_molecule.parquet', index=False)
LOSS_SUMMARY.merge(POLICY_CMP, on=['panel', 'external_source'], how='left').to_csv(REPORT / 'filter_loss_summary.csv', index=False)
RESCUED.to_parquet(REPORT / 'rescue_attempts.parquet', index=False)

In [ ]:
# mass-filter audit: truths outside the CURRENT external mass range (never changed automatically)
from casmi.validation.external_coverage import inference_mass_range, mass_range_audit
MASS_RANGE = mass_range_audit(MOL, UCFG.filters)
display(MASS_RANGE)
INF_PATH = COV.get('inference_metadata_path')
INF_RANGE = inference_mass_range(read_optional_table(P.drive_root / INF_PATH) if INF_PATH else None)
print('inference-safe neutral-mass range:', json.dumps(INF_RANGE, default=str))
MASS_RANGE.to_csv(REPORT / 'mass_range_audit.csv', index=False)
write_json(REPORT / 'inference_mass_range.json', INF_RANGE)

In [ ]:
# adduct / charge audit: only adducts PRESENT in the data; [M]+ style adducts surfaced, never assumed
from casmi.validation.external_coverage import adduct_charge_audit
ADDUCTS = adduct_charge_audit(Q.loc[Q['panel'] == 'dev_grouped', 'adduct']) if Q['adduct'].notna().any() else pd.DataFrame()
display(ADDUCTS.head(40))
PERM = ADDUCTS[ADDUCTS['permanently_charged_compatible']] if len(ADDUCTS) else ADDUCTS
print('adducts compatible with a PERMANENTLY charged candidate:', PERM['adduct'].tolist() if len(PERM) else 'none present in the data')
ADDUCTS.to_csv(REPORT / 'adduct_charge_audit.csv', index=False)

In [ ]:
# representative SMILES -> EXACT project canonicalizer -> connectivity key == stored key ? (resumable per bucket)
from casmi.validation.smiles_roundtrip import run_universe_roundtrip
RT = COV['roundtrip']
RT_FAILS, RT_SUMMARY = run_universe_roundtrip(UNIVERSE_ROOT, REPORT / 'roundtrip_parts', n_jobs=resolve_n_jobs(RT['n_jobs']),
                                              max_per_bucket=RT['max_per_bucket'], seed=CFG['validation']['split_seed'])
RT_FAILS.to_parquet(REPORT / 'smiles_roundtrip_failures.parquet', index=False)
write_json(REPORT / 'smiles_roundtrip_summary.json', RT_SUMMARY)
print(json.dumps({k: RT_SUMMARY[k] for k in ('n_candidates', 'n_roundtrip_ok', 'n_roundtrip_fail', 'failure_rate')}, indent=2))
display(pd.DataFrame(RT_SUMMARY['by_stratum']).T)

In [ ]:
# official-metric parity: only with a COMPETITION-PROVIDED canonicalizer file (none in the repository)
from casmi.validation.external_coverage import official_metric_parity
from casmi.validation.smiles_roundtrip import project_keys
OM = COV.get('official_metric') or {}
OM_PATH = (P.drive_root / OM['module_path']) if OM.get('module_path') else None
SAMPLE = (TRUTH_ROWS.sort_values('connectivity_key')['representative_smiles'].head(int(OM.get('sample_size') or 500)).tolist()
          + RT_FAILS['representative_smiles'].head(100).tolist())          # deterministic sample + round-trip failures
OFFICIAL = official_metric_parity(OM_PATH, OM.get('canonicalizer_function'), SAMPLE, project_keyfn=lambda s: project_keys([s])[0][0])
write_json(REPORT / 'official_metric_parity.json', OFFICIAL)
print('OFFICIAL_METRIC_PARITY =', OFFICIAL['OFFICIAL_METRIC_PARITY'], '|', OFFICIAL.get('reason', ''))

In [ ]:
# public-notebook baseline: coverage gap vs ranking gap (LOCAL artifacts only; none found in the repository)
from casmi.validation.external_coverage import public_baseline_coverage
PB = COV.get('public_baseline') or {}
PUB = read_optional_table(P.drive_root / PB['path']) if PB.get('path') else None
PUB_STATUS, PUB_CMP = public_baseline_coverage(PUB, Q if PUB is not None else None, WIN, ppm=PB.get('ppm', 5),
                                               keyfn=lambda s: project_keys([s])[0][0])
write_json(REPORT / 'public_baseline_coverage.json', PUB_STATUS)
PUB_CMP.to_csv(REPORT / 'public_baseline_coverage_comparison.csv', index=False)
print('PUBLIC_BASELINE_COVERAGE =', PUB_STATUS['PUBLIC_BASELINE_COVERAGE'], '|', PUB_STATUS.get('reason', ''))

In [ ]:
# ESTIMATE_ONLY planning helper: linear extrapolation of the OBSERVED COCONUT Stage-A throughput (not a measurement)
from casmi.validation.external_coverage import ESTIMATE_ONLY, estimate_runtime, observed_stage_a_throughput
TP = {s: observed_stage_a_throughput(v['namespace'], s) for s, v in ACTIVE.items() if v['status'] == 'complete'}
print(json.dumps(TP, indent=2))
rps = (TP.get('COCONUT') or {}).get('records_per_second')
EST = ({f'{n:,}_records': estimate_runtime(n, rps) for n in (1_000_000, 10_000_000, 120_000_000)} if rps
       else {'label': ESTIMATE_ONLY, 'status': 'no observed throughput in the Stage-A markers'})
print(ESTIMATE_ONLY, '(same worker count as the observed run):')
print(json.dumps(EST, indent=2, default=str))

In [ ]:
# mechanical evaluation of the PRE-DECLARED rules + gate summary (consumed by notebook 14)
from casmi.validation.external_coverage import decision_metrics, evaluate_decision_rules, protocol_validity
PROTO = protocol_validity(USTATUS, MSTATUS, Q)
METRICS = decision_metrics(REACH_PANEL, POOL, LOSS_SUMMARY, POLICY)
DECISION = evaluate_decision_rules(METRICS, POLICY, protocol_valid=PROTO['protocol_valid'])
write_json(REPORT / 'decision_rules.json', {**DECISION_RULES, 'evaluation': DECISION, 'metrics': METRICS})
GATE = {'notebook': NOTEBOOK, 'coverage_gate_version': COVERAGE_GATE_VERSION, 'completed': True, **PROTO,
        'decision': DECISION['decision'], 'fired_rules': DECISION['fired_rules'], 'metrics': METRICS,
        'universe': {'status': USTATUS['status'], 'n_candidates': USTATUS.get('n_candidates'), 'build_id': MSTATUS.get('build_id'),
                     'parent_stage_b_build_id': MSTATUS.get('parent_stage_b_build_id')},
        'stage_a_builds': {k: v['build_id'] for k, v in ACTIVE.items()}, 'pubchem_membership': PC_SUMMARY.get('status'),
        'official_metric_parity': OFFICIAL['OFFICIAL_METRIC_PARITY'], 'public_baseline_coverage': PUB_STATUS['PUBLIC_BASELINE_COVERAGE'],
        'smiles_roundtrip_failure_rate': RT_SUMMARY['failure_rate'], 'reachability_by_panel': REACH_PANEL.to_dict(orient='records'),
        'ranking_metrics': 'NOT COMPUTED (coverage gate only; ranking starts in notebook 14)',
        'finished_at': datetime.now(timezone.utc).isoformat()}
write_json(REPORT / 'coverage_gate_summary.json', GATE)
print('DECISION:', GATE['decision'], '| fired:', GATE['fired_rules'], '| protocol valid:', GATE['protocol_valid'], GATE['protocol_reasons'])
for r, v in DECISION['rules'].items():
    print('  Rule', r, '|', v['status'], '|', v['reason'])

In [ ]:
# figures 1-2: external reachability by panel (COCONUT / PubChem / union); incremental PubChem over COCONUT by mass bin
import matplotlib.pyplot as plt
from casmi.validation.external_coverage import mass_bin_labels
FIG = REPORT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
ORDER = mass_bin_labels(CFG['validation']['mass_bins_da']) + ['unknown']
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
REACH_PANEL.set_index('panel')[['coconut_reachability', 'pubchem_reachability', 'union_external_reachability']].plot.bar(
    ax=ax[0], title='external reachability by panel (TRAIN never counts)')
ax[0].set_ylim(0, 1)
for p, g in REACH_MASS.groupby('panel'):
    g = g.set_index('mass_bin').reindex([o for o in ORDER if o in set(g['mass_bin'])])
    ax[1].plot(g.index, g['incremental_pubchem_over_coconut_pp'], marker='o', label=p)
ax[1].set_title('incremental PubChem reachability over COCONUT (pp) by mass bin')
ax[1].tick_params(axis='x', rotation=45)
ax[1].legend()
fig.tight_layout()
fig.savefig(FIG / 'external_reachability.png', dpi=120)

In [ ]:
# figures 3-4: candidate-pool sizes at 2 / 5 / 10 ppm; current finalized vs projected PubChem Tier-1 (RAW) density
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for ppm, g in WIN[WIN['panel'] == 'dev_grouped'].groupby('ppm'):
    ax[0].hist(np.log10(g['full_current_universe_pool_size'].clip(lower=1)), bins=60, histtype='step', label=f'{ppm} ppm')
ax[0].set_xlabel('log10 pool size (TRAIN u COCONUT, finalized connectivities)')
ax[0].set_title('candidate-pool sizes (dev_grouped)')
ax[0].legend()
d = POOL[(POOL['n'] > 0) & POOL['pool'].isin(['full_current_universe_pool_size', 'projected_pubchem_tier1_pool_size'])]
if len(d):
    d.pivot_table(index=['panel', 'ppm'], columns='pool', values='median').plot.bar(
        ax=ax[1], logy=True, title='median pool: finalized connectivities vs projected PubChem Tier-1 RAW CIDs')
fig.tight_layout()
fig.savefig(FIG / 'pool_density.png', dpi=120)

In [ ]:
# figures 5-6: filter-loss / rescue waterfall (truth counts) and representative-SMILES round trip by stratum
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ls = LOSS_SUMMARY[LOSS_SUMMARY['panel'] == 'dev_grouped']
if len(ls):
    s = ls[['reachable_before_filters', 'lost_count', 'rescued_count', 'newly_reachable_by_rescue', 'reachable_after_rescue']].sum()
    steps = {'reachable before filters': s['reachable_before_filters'], 'lost (current policy)': -s['lost_count'],
             'rescued (rescue policy)': s['rescued_count'], 'newly reachable (rescue)': s['newly_reachable_by_rescue'],
             'reachable after rescue': s['reachable_after_rescue']}
    ax[0].bar(list(steps), list(steps.values()), color=['grey', 'tab:red', 'tab:green', 'tab:blue', 'black'])
    ax[0].set_title('filter loss / rescue (dev_grouped truths)')
    ax[0].tick_params(axis='x', rotation=20)
rt = pd.DataFrame(RT_SUMMARY['by_stratum']).T
if len(rt):
    rt[['n_ok', 'n_fail']].astype(float).plot.bar(stacked=True, ax=ax[1], logy=True, title='representative-SMILES round trip by stratum')
fig.tight_layout()
fig.savefig(FIG / 'filter_loss_and_roundtrip.png', dpi=120)

In [ ]:
# artifact metadata + experiment log (no ranking metric is logged: none was computed)
from casmi.workspace.artifacts import write_metadata
from casmi.workspace.experiments import log_experiment
write_metadata(REPORT, NOTEBOOK, config={'coverage': COV, 'universe_filters': CFG['universe']['filters']}, seed=SEED, repo_root=REPO_ROOT,
               input_paths={'universe': UNIVERSE_ROOT, 'universe_work': WORK, 'dev_queries': dev_path, 'regimes': REG_PATH,
                            'pubchem_membership': PC_DIR}, schema_version=COVERAGE_GATE_VERSION,
               extra={'decision': GATE['decision'], 'universe_build_id': MSTATUS.get('build_id'), 'policy_identity': POLICY.identity(),
                      'stage_a_builds': GATE['stage_a_builds']})
np_row = REACH_PANEL.set_index('panel')
note = f"{GATE['decision']} | coconut reachability (dev_grouped) = {np_row['coconut_reachability'].get('dev_grouped')}"
log_experiment(P.experiments_dir, 'gateA_external_coverage', 'external candidate coverage gate (notebook 13; coverage only)', ['coverage'],
               config_path=CFG['_config_path'], notes=note)
print('saved under', REPORT)
for f in sorted(REPORT.glob('*')):
    print(' ', f.name)

## Final decision summary

The `c-decision` cell prints the mechanical decision from `coverage_gate_summary.json`:

| decision | next step |
|---|---|
| `BUILD_PUBCHEM_TIER1_NEXT` | build the PubChem Tier-1 table (`casmi.candidates.pubchem_tier1`, set `coverage.pubchem_tier1`), then a deliberate new external universe |
| `REBUILD_COCONUT_WITH_RESCUE_POLICY` | set `universe.cleanup_policy: rescue_policy` and re-run notebook 12. A new Stage-A build plus Stage B / C follow; old builds are kept |
| `BUILD_FORMULA_PRUNING_BEFORE_RANKING` | pool density is the bottleneck: formula / pruning work before any ranker |
| `PROCEED_TO_ANALOG_BASELINE` | run `14_colab_analog_baseline.ipynb` (it checks this summary first) |
| `UNDETERMINED_THRESHOLDS_OR_INPUTS_MISSING` | set the missing `coverage.decision` thresholds **before** re-running (not after looking at results) |
| `PROTOCOL_INVALID` | the finalized universe has no external source: fix notebook 12 inputs first |

Notebook 14 refuses to run unless `completed`, `protocol_valid` and the decision permits the analog baseline.